In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mxlpy import (
    InitialAssignment,
    SteadyStateModelBuilder,
    cartesian_product,
    fit,
    plot,
    scan,
)
from mxlpy.meta import generate_model_code_mxlweb

# Steady-state models

A `SteadyStateModelBuilder` describes a model without any dynamics: there are no variables, only parameters and quantities derived from them.
This makes it a good fit for algebraic models, e.g. analytic steady-state solutions, response curves or allometric relationships.

Because there is nothing to integrate, a `SteadyStateModelBuilder` is never simulated.
Instead it is *evaluated* directly, which is both simpler and much faster.

As an example we use an enzyme following Michaelis-Menten kinetics, which is competitively inhibited by $I$

$$ v = \frac{V_{\max} \cdot S}{K_m \left(1 + \frac{I}{K_i}\right) + S} \qquad V_{\max} = k_{cat} \cdot E_{tot}$$

## Building the model

Parameters and derived quantities work just like in the other builders.
Parameters can also be initial assignments, and derived quantities can depend on each other in any order.

In [ ]:
def mul(a: float, b: float) -> float:
    return a * b


def apparent_km(km: float, i: float, ki: float) -> float:
    return km * (1 + i / ki)


def michaelis_menten(s: float, vmax: float, km: float) -> float:
    return vmax * s / (km + s)


def ratio(a: float, b: float) -> float:
    return a / b


def get_model() -> SteadyStateModelBuilder:
    return (
        SteadyStateModelBuilder()
        .add_parameters(
            {"kcat": 10.0, "e_tot": 0.5, "km": 2.0, "ki": 1.0, "S": 1.0, "I": 0.0}
        )
        .add_parameter("vmax", InitialAssignment(fn=mul, args=["kcat", "e_tot"]))
        .add_derived("km_app", fn=apparent_km, args=["km", "I", "ki"])
        .add_derived("v", fn=michaelis_menten, args=["S", "vmax", "km_app"])
        .add_derived("saturation", fn=ratio, args=["v", "vmax"])
    )


model = get_model()
model.derived

## Evaluation

`evaluate` returns all derived values as a `pandas.Series`.
This replaces the `Simulator` for this builder.

In [ ]:
model.evaluate()

Optionally, the parameters (including initial assignments) can be included as well

In [ ]:
model.evaluate(include_parameters=True)

The model is a normal fluent builder, so you can update parameters and evaluate again

In [ ]:
model.update_parameter("I", 2.0).evaluate()

## Parameter scans

`scan.steady_state` accepts a `SteadyStateModelBuilder` and evaluates it for every row of `to_scan`.
It returns an `EvaluationScan`, which holds the evaluated values by scanned value.

Since nothing is integrated, simulation specific arguments like `integrator`, `y0` or `tolerance` don't apply here.
Time course and protocol scans are not available for this builder.

In [ ]:
res = scan.steady_state(
    get_model(),
    to_scan=pd.DataFrame({"S": np.geomspace(0.1, 100, 21)}),
)

fig, ax = plot.lines(res.combined.loc[:, ["v"]])
ax.set(xscale="log", xlabel="substrate / a.u.", ylabel="rate / a.u.")
plt.show()

`get_args` can additionally include the parameters

In [ ]:
res.get_args(include_parameters=True).head()

Scanning multiple parameters works via `cartesian_product`, which yields a `pandas.MultiIndex`

In [ ]:
res = scan.steady_state(
    get_model(),
    to_scan=cartesian_product(
        {
            "S": np.geomspace(0.1, 100, 10),
            "I": np.linspace(0, 5, 6),
        }
    ),
)

fig, ax, _ = plot.heatmap_from_2d_idx(res.combined, "v")
ax.set(xlabel="S", ylabel="I", title="rate v")
plt.show()

If an evaluation fails with an arithmetic error (e.g. a division by zero), the respective row is filled with `NaN` instead of failing the whole scan

In [ ]:
scan.steady_state(
    get_model(),
    to_scan=pd.DataFrame({"ki": [1.0, 0.0, 2.0]}),
).combined

## Fitting

`fit.steady_state` accepts a `SteadyStateModelBuilder` and fits its parameters to a `pandas.Series` of data.
The data can contain any parameter or derived value.

Here we assume we measured the rate $v$ and the saturation of the enzyme at $S = 2$ and fit $K_m$ and $k_{cat}$.

In [ ]:
fitted = fit.steady_state(
    get_model().update_parameter("S", 2.0),
    p0={"km": 1.0, "kcat": 5.0},
    data=pd.Series({"v": 1.25, "saturation": 0.25}),
    minimizer=fit.LocalScipyMinimizer(),
).unwrap_or_err()
fitted.best_pars

The returned `Fit` contains a copy of the model, which can be evaluated with the best parameters

In [ ]:
fitted.model.update_parameters(fitted.best_pars).evaluate()

## Export

`SteadyStateModelBuilder` models can be saved to the native `.mxl.json` format with `mxlpy.save` / `mxlpy.load`, and exported to [MxlWeb](https://github.com/Computational-Biology-Aachen/mxl-web), which evaluates them in the browser.

In [ ]:
print(generate_model_code_mxlweb(get_model()))

## Current limitations

The `SteadyStateModelBuilder` currently supports `evaluate`, `scan.steady_state` and `fit.steady_state`.
Other routines (e.g. `mc`, `mca`, group, ensemble or joint fits) only accept the integrable builders so far.